## Pengaplikasian Genetic Algorithm dalam Optimasi Barisan Probe DNA

In [ ]:
import random

# --- Parameter Umum ---
loop_len = 30  # Panjang loop yang diinginkan
fixed_left_loop = "TATAAGCACTTTA"  # Urutan 13 nukleotida tetap untuk bagian kiri loop spesifik terhadap miRNA 20a
flexible_len = loop_len - len(fixed_left_loop) # Panjang bagian loop yang akan dibuat secara acak
stem_len = 18  # Panjang stem (batang) yang diinginkan (baik maju maupun mundur)
pop_size = 200  # Ukuran populasi untuk algoritma genetika
max_gen = 400  # Jumlah generasi maksimum untuk algoritma genetika
mutation_rate = 0.3  # Probabilitas terjadinya mutasi selama algoritma genetika

# --- Fungsi Energi dan Validasi ---
def calculate_delta_g(full_sequence):
    nn_energy = {
        # Nilai energi nearest-neighbor (kcal/mol) untuk pasangan basa DNA
        "AA": -1.00, "TT": -0.57, "AT": -0.88, "TA": -0.58,
        "CA": -1.45, "TG": -1.45, "GT": -1.44, "AC": -1.44,
        "CT": -1.28, "AG": -1.28, "GA": -1.30, "TC": -1.30,
        "CG": -2.17, "GC": -2.24, "GG": -1.84, "CC": -1.84
    }
    total_dg = 0
    # Menjumlahkan delta G untuk setiap pasangan basa yang berdekatan
    for i in range(len(full_sequence) - 1):
        pair = full_sequence[i:i+2]
        total_dg += nn_energy.get(pair, 0)  # Mengambil energi, 0 jika pasangan tidak ditemukan
    return total_dg

def gc_content(sequence):
    # Menghitung kandungan Guanin-Sitosin dalam persentase
    gc = sequence.count('G') + sequence.count('C')
    return (gc / len(sequence)) * 100

def has_repetition(sequence, max_repeat=3):
    # Memeriksa pengulangan basa berturut-turut melebihi max_repeat
    count = 1
    for i in range(1, len(sequence)):
        if sequence[i] == sequence[i-1]:
            count += 1
            if count > max_repeat:
                return True
        else:
            count = 1
    return False

def reverse_complement(seq):
    # Menghasilkan komplemen terbalik dari urutan DNA
    comp = {'A': 'T', 'T': 'A', 'G': 'C', 'C': 'G'}
    return ''.join(comp[b] for b in reversed(seq))  # Membalik dan mengkomplemen setiap basa

def has_complementary_pair(loop_seq, min_len=2, max_len=7):
    # Memeriksa apakah urutan loop mengandung pasangan komplementer internal (misalnya, pembentukan jepit rambut)
    comp = {'A': 'T', 'T': 'A', 'G': 'C', 'C': 'G'}
    n = len(loop_seq)
    seen = set()  # Menyimpan sub-urutan yang telah ditemui
    for length in range(min_len, max_len + 1):
        for i in range(n - length + 1):
            subseq = loop_seq[i:i+length]
            rc_subseq = ''.join(comp.get(b, 'N') for b in reversed(subseq))  # Mendapatkan komplemen terbalik
            if rc_subseq in seen:  # Jika komplemen terbalik dari sub-urutan saat ini telah terlihat, itu adalah pasangan
                return True
            seen.add(subseq)
    return False

def random_dna(n):
    # Menghasilkan urutan DNA acak sepanjang n
    return ''.join(random.choices("ACGT", k=n))

def calculate_dg_components(full_sequence, stem_len=18, loop_len=30):
    # Menghitung delta G untuk komponen stem dan loop, termasuk penalti AT
    stem_seq_left = full_sequence[:stem_len]
    stem_seq_right = full_sequence[-stem_len:]
    dg_stem = calculate_delta_g(stem_seq_left)  # Delta G dari stem kiri

    at_penalty = 0
    # Menambahkan penalti jika pasangan basa pertama/terakhir dari stem adalah A-T
    if (stem_seq_left[0], stem_seq_right[-1]) in [('A', 'T'), ('T', 'A')]:
        at_penalty += 0.5
    if (stem_seq_left[-1], stem_seq_right[0]) in [('A', 'T'), ('T', 'A')]:
        at_penalty += 0.5

    dg_loop = 6.3  # Kontribusi delta G tetap untuk loop
    total_dg = dg_stem + dg_loop + at_penalty  # Total delta G untuk probe
    return total_dg, dg_stem, dg_loop, at_penalty

def genetic_stem():
    # Algoritma genetika untuk menemukan urutan stem yang optimal
    def random_candidate(length=8, fixed_start="GTGCAGGTAG"):
        # Menghasilkan kandidat stem acak, memastikan konten GC yang valid dan tidak ada pengulangan
        while True:
            seq = ''.join(random.choices('ACGT', k=length))
            full_seq = fixed_start + seq
            if 40 <= gc_content(full_seq) <= 60 and not has_repetition(full_seq):
                return seq

    def crossover(p1, p2):
        # Melakukan crossover satu titik antara dua urutan induk
        point = random.randint(1, len(p1)-1)
        return p1[:point] + p2[point:]

    def mutate(seq, mutation_rate=0.3):
        # Memutasikan urutan dengan mengubah basa secara acak
        seq = list(seq)
        for i in range(len(seq)):
            if random.random() < mutation_rate:
                seq[i] = random.choice([b for b in "ACGT" if b != seq[i]])  # Mengubah ke basa yang berbeda
        return ''.join(seq)

    def fitness(seq, fixed_start="GTGCAGGTAG", dg_target=-20.0):
        # Menghitung kebugaran untuk kandidat stem: lebih dekat ke target delta G, penalti lebih rendah untuk GC/pengulangan yang tidak valid
        full_seq = fixed_start + seq
        mean_dg = calculate_delta_g(full_seq)
        gc = gc_content(full_seq)
        penalty = 0
        if not (40 <= gc <= 60): penalty += 20  # Penalti untuk konten GC di luar jangkauan
        if has_repetition(full_seq): penalty += 20  # Penalti untuk pengulangan
        return -abs(mean_dg - dg_target) - penalty  # Negatif dari perbedaan absolut dari target DG

    fixed_start = "GTGCAGGTAG"  # Bagian tetap dari stem
    population = [random_candidate(8, fixed_start) for _ in range(200)]  # Menginisialisasi populasi
    seen = set()  # Untuk menyimpan kandidat valid yang unik
    best_pool = []  # Kumpulan kandidat terbaik

    for _ in range(400):  # Berulang melalui generasi
        population = sorted(population, key=lambda x: fitness(x, fixed_start), reverse=True)  # Mengurutkan berdasarkan kebugaran
        next_gen = population[:10]  # Elitisme: mempertahankan 10 kandidat teratas

        for seq in population[:25]:  # Mempertimbangkan 25 teratas untuk best_pool
            if seq not in seen:
                full = fixed_start + seq
                if 40 <= gc_content(full) <= 60 and not has_repetition(full):
                    seen.add(seq)
                    best_pool.append(seq)

        while len(next_gen) < 200:  # Mengisi sisa generasi berikutnya
            p1, p2 = random.choices(population[10:35], k=2)  # Memilih induk dari 10-35 teratas
            child = crossover(p1, p2)  # Melakukan crossover
            child = mutate(child)  # Memutasikan anak
            full_child = fixed_start + child
            if 40 <= gc_content(full_child) <= 60 and not has_repetition(full_child):  # Memvalidasi anak
                next_gen.append(child)
        population = next_gen  # Memperbarui populasi

    best = sorted(set(best_pool), key=lambda x: fitness(x, fixed_start), reverse=True)[0]  # Mendapatkan stem terbaik
    return fixed_start + best  # Mengembalikan stem lengkap yang optimal

def generate_probe_candidates():
    # Fungsi utama untuk menghasilkan kandidat probe lengkap (stem-loop-stem)
    pop = [random_dna(flexible_len) for _ in range(pop_size)]  # Menginisialisasi populasi loop

    def loop_fitness(seq):
        # Menghitung kebugaran untuk kandidat loop berdasarkan konten GC, pengulangan, dan pasangan komplementer
        loop_seq = fixed_left_loop + seq
        fitness = 0

        if not (40 <= gc_content(loop_seq) <= 60):
            fitness += -20  # Penalti untuk GC di luar jangkauan
        if has_repetition(loop_seq):
            fitness += -20  # Penalti untuk pengulangan
        if has_complementary_pair(loop_seq, min_len=2, max_len=7):
            fitness += -20  # Penalti untuk pasangan komplementer internal
        if loop_seq[-1] == 'A':
            fitness += -50  # Penalti besar jika basa terakhir loop adalah 'A'

        return fitness

    loop_scores = {}  # Untuk menyimpan skor fitness loop (tidak langsung digunakan untuk seleksi dalam GA, tetapi untuk logging/debugging)
    for gen in range(max_gen):  # Algoritma genetika untuk optimasi loop
        pop = sorted(pop, key=loop_fitness, reverse=True)  # Mengurutkan populasi berdasarkan fitness loop
        next_gen = pop[:10]  # Elitisme: mempertahankan 10 kandidat loop teratas
        for candidate in next_gen:
            loop_scores[candidate] = loop_fitness(candidate)  # Menyimpan fitness

        while len(next_gen) < pop_size:  # Mengisi sisa generasi berikutnya
            p1, p2 = random.choices(pop[:25], k=2)  # Memilih induk dari 25 teratas
            cut = random.randint(1, flexible_len - 2)  # Titik crossover
            child = p1[:cut] + p2[cut:]  # Crossover
            child = ''.join(
                random.choice("ACGT") if random.random() < mutation_rate else c
                for c in child
            )  # Memutasikan anak
            next_gen.append(child)  # Menambahkan anak ke generasi berikutnya
        pop = next_gen  # Memperbarui populasi

    stem_forward = genetic_stem()  # Menghasilkan stem maju menggunakan GA-nya
    stem_reverse = reverse_complement(stem_forward)  # Mendapatkan komplemen terbalik dari stem maju

    # Peringkat 5 kandidat probe gabungan teratas
    ranked = []
    for seq in set(pop):  # Berulang melalui kandidat loop unik dari populasi akhir
        loop_seq = fixed_left_loop + seq
        full_probe = stem_forward + loop_seq + stem_reverse  # Membuat probe lengkap
        total_dg, dg_stem, dg_loop, dg_at = calculate_dg_components(full_probe)  # Menghitung komponen DG
        loop_fit = loop_fitness(seq)  # Mendapatkan kebugaran loop
        total_score = loop_fit - total_dg  # Menggabungkan kebugaran loop dan total DG untuk skor keseluruhan (total_dg yang lebih rendah lebih baik, jadi dikurangi)
        ranked.append((total_score, seq, full_probe, loop_seq, total_dg, dg_stem, dg_loop, dg_at))

    ranked = sorted(ranked, reverse=True)[:5]  # Mengurutkan berdasarkan total skor dan mengambil 5 teratas

    print("Top 5 kandidat probe lengkap (stem-loop-stem):")
    for i, (score, seq, full_probe, loop_seq, total_dg, dg_stem, dg_loop, dg_at) in enumerate(ranked, 1):
        print(f"\n{i}. Full Probe:")
        print(f"5'-{full_probe}-3'")
        print(f"Stem Forward : {stem_forward}")
        print(f"Loop         : {loop_seq}")
        print(f"Stem Reverse : {stem_reverse}")
        print(f"Length: {len(full_probe)} nt | GC: {gc_content(full_probe):.2f}%")
        print(f"ΔG total: {total_dg:.2f} kcal/mol = stem: {dg_stem:.2f} + loop: {dg_loop:.2f} + AT: {dg_at:.2f}")

# --- Jalankan Fungsi ---
generate_probe_candidates()  # Menjalankan proses pembuatan probe

Top 5 kandidat probe lengkap (stem-loop-stem):

1. Full Probe:
5'-GTGCAGGTAGTTCTTTAGTATAAGCACTTTATTCACTACGTAGGCCCTCTAAAGAACTACCTGCAC-3'
Stem Forward : GTGCAGGTAGTTCTTTAG
Loop         : TATAAGCACTTTATTCACTACGTAGGCCCT
Stem Reverse : CTAAAGAACTACCTGCAC
Length: 66 nt | GC: 42.42%
ΔG total: -14.29 kcal/mol = stem: -20.59 + loop: 6.30 + AT: 0.00

2. Full Probe:
5'-GTGCAGGTAGTTCTTTAGTATAAGCACTTTATTCAATACGGGAGGACGCTAAAGAACTACCTGCAC-3'
Stem Forward : GTGCAGGTAGTTCTTTAG
Loop         : TATAAGCACTTTATTCAATACGGGAGGACG
Stem Reverse : CTAAAGAACTACCTGCAC
Length: 66 nt | GC: 42.42%
ΔG total: -14.29 kcal/mol = stem: -20.59 + loop: 6.30 + AT: 0.00

3. Full Probe:
5'-GTGCAGGTAGTTCTTTAGTATAAGCACTTTATGCGTCCTGTGGGCCGCCTAAAGAACTACCTGCAC-3'
Stem Forward : GTGCAGGTAGTTCTTTAG
Loop         : TATAAGCACTTTATGCGTCCTGTGGGCCGC
Stem Reverse : CTAAAGAACTACCTGCAC
Length: 66 nt | GC: 48.48%
ΔG total: -14.29 kcal/mol = stem: -20.59 + loop: 6.30 + AT: 0.00

4. Full Probe:
5'-GTGCAGGTAGTTCTTTAGTATAAGCACTTTATCTGTCCTTGTTGCCAGC